In [2]:
import pandas as pd
import numpy as np
import joblib
import sys
import os

sys.path.append("../src")

from preprocessing import (
    create_multiclass_target,
    drop_failure_columns,
    build_binary_dataset,
    build_multiclass_dataset,
    split_data,
    build_preprocessing_pipeline
)

from feature_engineering import create_engineered_features

In [3]:
df = pd.read_csv("../data/interim/cleaned_data.csv")

print("Shape:", df.shape)
df.head()

Shape: (10000, 12)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
df = create_multiclass_target(df)

df["Failure Type"].value_counts()

Failure Type
No Failure          9652
HDF                  106
PWF                   80
OSF                   78
TWF                   42
Multiple Failure      24
RNF                   18
Name: count, dtype: int64

In [5]:
df = drop_failure_columns(df)

df.columns

Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
       'Machine failure', 'Failure Type'],
      dtype='str')

In [6]:
df = create_engineered_features(df)

df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,Failure Type,Temp_diff,Power
0,M,298.1,308.6,1551,42.8,0,0,No Failure,10.5,66382.8
1,L,298.2,308.7,1408,46.3,3,0,No Failure,10.5,65190.4
2,L,298.1,308.5,1498,49.4,5,0,No Failure,10.4,74001.2
3,L,298.2,308.6,1433,39.5,7,0,No Failure,10.4,56603.5
4,L,298.2,308.7,1408,40.0,9,0,No Failure,10.5,56320.0


In [28]:
# At this point df contains:
# - Sensor features
# - Engineered features
# - Machine failure
# - Failure Type

df_full = df.copy()

print("Full dataset shape:", df_full.shape)
df_full.head()

Full dataset shape: (10000, 10)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,Failure Type,Temp_diff,Power
0,M,298.1,308.6,1551,42.8,0,0,No Failure,10.5,66382.8
1,L,298.2,308.7,1408,46.3,3,0,No Failure,10.5,65190.4
2,L,298.1,308.5,1498,49.4,5,0,No Failure,10.4,74001.2
3,L,298.2,308.6,1433,39.5,7,0,No Failure,10.4,56603.5
4,L,298.2,308.7,1408,40.0,9,0,No Failure,10.5,56320.0


In [29]:
binary_full = df_full.drop(columns=["Failure Type"])

print("Binary full dataset shape:", binary_full.shape)
binary_full.head()

Binary full dataset shape: (10000, 9)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,Temp_diff,Power
0,M,298.1,308.6,1551,42.8,0,0,10.5,66382.8
1,L,298.2,308.7,1408,46.3,3,0,10.5,65190.4
2,L,298.1,308.5,1498,49.4,5,0,10.4,74001.2
3,L,298.2,308.6,1433,39.5,7,0,10.4,56603.5
4,L,298.2,308.7,1408,40.0,9,0,10.5,56320.0


In [30]:
multiclass_full = df_full.drop(columns=["Machine failure"])

print("Multiclass full dataset shape:", multiclass_full.shape)
multiclass_full.head()

Multiclass full dataset shape: (10000, 9)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Failure Type,Temp_diff,Power
0,M,298.1,308.6,1551,42.8,0,No Failure,10.5,66382.8
1,L,298.2,308.7,1408,46.3,3,No Failure,10.5,65190.4
2,L,298.1,308.5,1498,49.4,5,No Failure,10.4,74001.2
3,L,298.2,308.6,1433,39.5,7,No Failure,10.4,56603.5
4,L,298.2,308.7,1408,40.0,9,No Failure,10.5,56320.0


In [31]:
os.makedirs("../data/processed", exist_ok=True)

binary_full.to_csv(
    "../data/processed/binary_full_dataset.csv",
    index=False
)

multiclass_full.to_csv(
    "../data/processed/multiclass_full_dataset.csv",
    index=False
)

print("Full datasets saved successfully.")

Full datasets saved successfully.


In [7]:
X_binary, y_binary = build_binary_dataset(df)
X_multi, y_multi = build_multiclass_dataset(df)

print("Binary Shape:", X_binary.shape)
print("Multiclass Shape:", X_multi.shape)

Binary Shape: (10000, 8)
Multiclass Shape: (10000, 8)


In [8]:
Xb_train, Xb_test, yb_train, yb_test = split_data(X_binary, y_binary)
Xm_train, Xm_test, ym_train, ym_test = split_data(X_multi, y_multi)

print("Binary Train:", Xb_train.shape)
print("Binary Test:", Xb_test.shape)

print("Multiclass Train:", Xm_train.shape)
print("Multiclass Test:", Xm_test.shape)

Binary Train: (7000, 8)
Binary Test: (3000, 8)
Multiclass Train: (7000, 8)
Multiclass Test: (3000, 8)


In [9]:
preprocessor = build_preprocessing_pipeline(Xb_train)

In [10]:
Xb_train_processed = preprocessor.fit_transform(Xb_train)
Xb_test_processed = preprocessor.transform(Xb_test)

Xm_train_processed = preprocessor.transform(Xm_train)
Xm_test_processed = preprocessor.transform(Xm_test)

In [11]:
os.makedirs("../data/processed", exist_ok=True)

joblib.dump(preprocessor, "../data/processed/preprocessor.pkl")

pd.to_pickle(
    (Xb_train_processed, Xb_test_processed, yb_train, yb_test),
    "../data/processed/binary_dataset.pkl"
)

pd.to_pickle(
    (Xm_train_processed, Xm_test_processed, ym_train, ym_test),
    "../data/processed/multiclass_dataset.pkl"
)

In [12]:
# Reload saved datasets
binary_data = pd.read_pickle("../data/processed/binary_dataset.pkl")
multiclass_data = pd.read_pickle("../data/processed/multiclass_dataset.pkl")

Xb_train_p, Xb_test_p, yb_train, yb_test = binary_data
Xm_train_p, Xm_test_p, ym_train, ym_test = multiclass_data

print("Binary train shape:", Xb_train_p.shape)
print("Binary test shape:", Xb_test_p.shape)

print("Multiclass train shape:", Xm_train_p.shape)
print("Multiclass test shape:", Xm_test_p.shape)

Binary train shape: (7000, 10)
Binary test shape: (3000, 10)
Multiclass train shape: (7000, 10)
Multiclass test shape: (3000, 10)


In [13]:
print("Binary feature count:", Xb_train_p.shape[1])
print("Multiclass feature count:", Xm_train_p.shape[1])

Binary feature count: 10
Multiclass feature count: 10


In [14]:
feature_names = preprocessor.get_feature_names_out()

print("Total Features After Encoding:", len(feature_names))
print(feature_names)

Total Features After Encoding: 10
['cat__Type_H' 'cat__Type_L' 'cat__Type_M' 'num__Air temperature [K]'
 'num__Process temperature [K]' 'num__Rotational speed [rpm]'
 'num__Torque [Nm]' 'num__Tool wear [min]' 'num__Temp_diff' 'num__Power']


In [15]:
for col in feature_names:
    if any(failure in col for failure in ["TWF", "HDF", "PWF", "OSF", "RNF"]):
        print("Leakage detected:", col)

In [16]:
print("Binary Train Distribution:")
print(yb_train.value_counts(normalize=True))

print("\nBinary Test Distribution:")
print(yb_test.value_counts(normalize=True))

Binary Train Distribution:
Machine failure
0    0.966143
1    0.033857
Name: proportion, dtype: float64

Binary Test Distribution:
Machine failure
0    0.966
1    0.034
Name: proportion, dtype: float64


In [17]:
print("Multiclass Train Distribution:")
print(ym_train.value_counts(normalize=True))

print("\nMulticlass Test Distribution:")
print(ym_test.value_counts(normalize=True))

Multiclass Train Distribution:
Failure Type
No Failure          0.965143
HDF                 0.010571
PWF                 0.008000
OSF                 0.007857
TWF                 0.004143
Multiple Failure    0.002429
RNF                 0.001857
Name: proportion, dtype: float64

Multiclass Test Distribution:
Failure Type
No Failure          0.965333
HDF                 0.010667
PWF                 0.008000
OSF                 0.007667
TWF                 0.004333
Multiple Failure    0.002333
RNF                 0.001667
Name: proportion, dtype: float64


In [18]:
total_samples = len(y_binary)

print("Train ratio:", len(yb_train) / total_samples)
print("Test ratio:", len(yb_test) / total_samples)

Train ratio: 0.7
Test ratio: 0.3


In [19]:
Xb_train_df = pd.DataFrame(
    Xb_train_p,
    columns=feature_names,
    index=yb_train.index
)

Xb_train_df.head()

,cat__Type_H,cat__Type_L,cat__Type_M,num__Air temperature [K],num__Process temperature [K],num__Rotational speed [rpm],num__Torque [Nm],num__Tool wear [min],num__Temp_diff,num__Power
1888,0.0,0.0,1.0,297.8,307.4,1902.0,24.3,129.0,9.6,46218.6
4858,0.0,1.0,0.0,303.7,312.3,1349.0,51.0,105.0,8.6,68799.0
8990,0.0,1.0,0.0,297.2,307.9,1493.0,38.4,146.0,10.7,57331.2
4901,0.0,0.0,1.0,303.6,312.3,1630.0,32.4,223.0,8.7,52812.0
7957,1.0,0.0,0.0,300.9,311.9,2140.0,16.5,43.0,11.0,35310.0


In [20]:
Xb_train_df = pd.DataFrame(
    Xb_train_p,
    columns=feature_names,
    index=yb_train.index
)

Xb_train_df.head()

,cat__Type_H,cat__Type_L,cat__Type_M,num__Air temperature [K],num__Process temperature [K],num__Rotational speed [rpm],num__Torque [Nm],num__Tool wear [min],num__Temp_diff,num__Power
1888,0.0,0.0,1.0,297.8,307.4,1902.0,24.3,129.0,9.6,46218.6
4858,0.0,1.0,0.0,303.7,312.3,1349.0,51.0,105.0,8.6,68799.0
8990,0.0,1.0,0.0,297.2,307.9,1493.0,38.4,146.0,10.7,57331.2
4901,0.0,0.0,1.0,303.6,312.3,1630.0,32.4,223.0,8.7,52812.0
7957,1.0,0.0,0.0,300.9,311.9,2140.0,16.5,43.0,11.0,35310.0


In [21]:
Xb_test_df = pd.DataFrame(
    Xb_test_p,
    columns=feature_names,
    index=yb_test.index
)

Xb_test_df.head()

,cat__Type_H,cat__Type_L,cat__Type_M,num__Air temperature [K],num__Process temperature [K],num__Rotational speed [rpm],num__Torque [Nm],num__Tool wear [min],num__Temp_diff,num__Power
7809,0.0,0.0,1.0,300.0,311.4,1537.0,35.4,120.0,11.4,54409.8
9069,0.0,0.0,1.0,297.2,308.2,1678.0,28.1,133.0,11.0,47151.8
3170,0.0,0.0,1.0,300.4,309.7,1685.0,32.3,167.0,9.3,54425.5
2750,1.0,0.0,0.0,299.7,309.2,1456.0,40.6,182.0,9.5,59113.6
6986,0.0,1.0,0.0,300.7,311.1,1688.0,30.7,161.0,10.4,51821.6


In [22]:
Xm_train_df = pd.DataFrame(
    Xm_train_p,
    columns=feature_names,
    index=ym_train.index
)

Xm_train_df.head()

,cat__Type_H,cat__Type_L,cat__Type_M,num__Air temperature [K],num__Process temperature [K],num__Rotational speed [rpm],num__Torque [Nm],num__Tool wear [min],num__Temp_diff,num__Power
6534,0.0,1.0,0.0,301.3,310.6,1436.0,44.2,76.0,9.3,63471.2
2405,0.0,1.0,0.0,299.1,308.7,1498.0,38.9,181.0,9.6,58272.2
7495,0.0,1.0,0.0,300.3,311.9,1609.0,34.0,177.0,11.6,54706.0
8563,0.0,1.0,0.0,298.0,308.9,1500.0,36.7,104.0,10.9,55050.0
5008,1.0,0.0,0.0,303.7,312.8,1488.0,42.8,54.0,9.1,63686.4


In [23]:
Xm_test_df = pd.DataFrame(
    Xm_test_p,
    columns=feature_names,
    index=ym_test.index
)

Xm_test_df.head()

,cat__Type_H,cat__Type_L,cat__Type_M,num__Air temperature [K],num__Process temperature [K],num__Rotational speed [rpm],num__Torque [Nm],num__Tool wear [min],num__Temp_diff,num__Power
1167,0.0,1.0,0.0,297.0,308.1,1362.0,52.5,213.0,11.1,71505.0
4319,0.0,0.0,1.0,301.6,310.2,1411.0,45.1,54.0,8.6,63636.1
6220,0.0,1.0,0.0,301.1,310.8,1470.0,49.3,116.0,9.7,72471.0
8708,0.0,0.0,1.0,297.2,308.5,1364.0,54.0,43.0,11.3,73656.0
2240,0.0,1.0,0.0,299.2,308.5,1457.0,44.7,195.0,9.3,65127.9


In [24]:
binary_train_full = Xb_train_df.copy()
binary_train_full["Machine failure"] = yb_train

binary_train_full.head()

,cat__Type_H,cat__Type_L,cat__Type_M,num__Air temperature [K],num__Process temperature [K],num__Rotational speed [rpm],num__Torque [Nm],num__Tool wear [min],num__Temp_diff,num__Power,Machine failure
1888,0.0,0.0,1.0,297.8,307.4,1902.0,24.3,129.0,9.6,46218.6,0
4858,0.0,1.0,0.0,303.7,312.3,1349.0,51.0,105.0,8.6,68799.0,0
8990,0.0,1.0,0.0,297.2,307.9,1493.0,38.4,146.0,10.7,57331.2,0
4901,0.0,0.0,1.0,303.6,312.3,1630.0,32.4,223.0,8.7,52812.0,0
7957,1.0,0.0,0.0,300.9,311.9,2140.0,16.5,43.0,11.0,35310.0,0


In [25]:
multi_train_full = Xm_train_df.copy()
multi_train_full["Failure Type"] = ym_train

multi_train_full.head()

,cat__Type_H,cat__Type_L,cat__Type_M,num__Air temperature [K],num__Process temperature [K],num__Rotational speed [rpm],num__Torque [Nm],num__Tool wear [min],num__Temp_diff,num__Power,Failure Type
6534,0.0,1.0,0.0,301.3,310.6,1436.0,44.2,76.0,9.3,63471.2,No Failure
2405,0.0,1.0,0.0,299.1,308.7,1498.0,38.9,181.0,9.6,58272.2,No Failure
7495,0.0,1.0,0.0,300.3,311.9,1609.0,34.0,177.0,11.6,54706.0,No Failure
8563,0.0,1.0,0.0,298.0,308.9,1500.0,36.7,104.0,10.9,55050.0,No Failure
5008,1.0,0.0,0.0,303.7,312.8,1488.0,42.8,54.0,9.1,63686.4,No Failure


In [26]:
print("Binary Train Shape:", binary_train_full.shape)
print("Binary Columns:")
print(binary_train_full.columns)

Binary Train Shape: (7000, 11)
Binary Columns:
Index(['cat__Type_H', 'cat__Type_L', 'cat__Type_M', 'num__Air temperature [K]',
       'num__Process temperature [K]', 'num__Rotational speed [rpm]',
       'num__Torque [Nm]', 'num__Tool wear [min]', 'num__Temp_diff',
       'num__Power', 'Machine failure'],
      dtype='str')


In [27]:
print("Multiclass Train Shape:", multi_train_full.shape)
print("Multiclass Columns:")
print(multi_train_full.columns)

Multiclass Train Shape: (7000, 11)
Multiclass Columns:
Index(['cat__Type_H', 'cat__Type_L', 'cat__Type_M', 'num__Air temperature [K]',
       'num__Process temperature [K]', 'num__Rotational speed [rpm]',
       'num__Torque [Nm]', 'num__Tool wear [min]', 'num__Temp_diff',
       'num__Power', 'Failure Type'],
      dtype='str')
